In [ ]:
"""
NBFC Enterprise Risk Architecture & Data Pipeline
-------------------------------------------------
This script acts as the backend engine for a fictional Indian NBFC.
It generates a hyper-realistic 1,500-profile auto loan portfolio,
calculates FOIR and NPA probabilities, provisions a SQLite database,
and extracts a targeted 'Intervention Hitlist' for the Collections team.
"""

import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("INITIATING END-TO-END NBFC DATA PIPELINE...")

# ==========================================
# PHASE 1: HYPER-REALISTIC DATA ENGINEERING
# ==========================================
print("\n[1/4] Synthesizing Demographic and Asset Data...")
np.random.seed(42)
num_records = 1500

# Demographics
cibil_scores = np.random.normal(730, 45, num_records).clip(550, 850).astype(int)
annual_incomes = np.random.lognormal(mean=14.6, sigma=0.55, size=num_records).clip(450000, 3800000).astype(int)
employment = np.random.choice(
    ['Salaried', 'Self-Employed Professional', 'Self-Employed Non-Professional'],
    num_records, p=[0.60, 0.25, 0.15]
)

# Vehicle Assets (Indian Market 2026)
vehicle_catalog = {
    'Maruti Suzuki Swift (Hatchback)': 750000,
    'Hyundai Venue (Compact SUV)': 1100000,
    'Tata Nexon EV (Electric SUV)': 1650000,
    'Mahindra XUV700 (Premium SUV)': 2300000,
    'Toyota Innova Hycross (Hybrid MPV)': 2900000,
    'Kia Seltos (Mid-Size SUV)': 1750000,
    'Honda City (Premium Sedan)': 1500000
}
vehicle_models = list(vehicle_catalog.keys())
vehicle_probs = [0.25, 0.15, 0.15, 0.12, 0.08, 0.15, 0.10]
chosen_vehicles = np.random.choice(vehicle_models, num_records, p=vehicle_probs)
vehicle_prices = np.array([vehicle_catalog[v] for v in chosen_vehicles])

# Financials & Origination
ltv_ratios = np.random.uniform(0.75, 0.95, num_records)
loan_amounts = (vehicle_prices * ltv_ratios).astype(int)

rates = np.where(cibil_scores >= 780, np.random.uniform(8.5, 9.2, num_records),
        np.where(cibil_scores >= 720, np.random.uniform(9.5, 10.5, num_records),
        np.where(cibil_scores >= 650, np.random.uniform(11.0, 12.5, num_records),
                 np.random.uniform(13.0, 15.5, num_records)))).round(2)

tenure_months = np.random.choice([36, 48, 60, 72, 84], num_records, p=[0.1, 0.2, 0.4, 0.2, 0.1])
monthly_rate = (rates / 100) / 12
emis = (loan_amounts * monthly_rate * ((1 + monthly_rate)**tenure_months)) / (((1 + monthly_rate)**tenure_months) - 1)
foir_pct = (emis / (annual_incomes / 12)).round(3)

# ==========================================
# PHASE 2: PREDICTIVE RISK ALGORITHM
# ==========================================
print("[2/4] Executing NPA Probability Algorithm...")
# NPA Logic: High FOIR + Low CIBIL + Employment Volatility
npa_probability = (
    (foir_pct * 0.5) +
    ((850 - cibil_scores) / 850 * 0.4) +
    np.where(employment != 'Salaried', 0.1, 0)
).clip(0, 1).round(4)

risk_category = np.where(npa_probability > 0.65, 'High Risk (Intervene)',
                np.where(npa_probability > 0.45, 'Medium Risk (Monitor)', 'Low Risk (Safe)'))

# Master DataFrame
df_master = pd.DataFrame({
    'LoanID': range(900001, 900001 + num_records),
    'CustomerID': range(100001, 100001 + num_records),
    'CIBIL_Score': cibil_scores,
    'Annual_Income_INR': annual_incomes,
    'Employment_Type': employment,
    'Vehicle_Model': chosen_vehicles,
    'Loan_Amount_INR': loan_amounts,
    'Interest_Rate_Pct': rates,
    'Tenure_Months': tenure_months,
    'Monthly_EMI_INR': emis.astype(int),
    'FOIR_Pct': foir_pct,
    'NPA_Probability': npa_probability,
    'Risk_Category': risk_category
})

# ==========================================
# PHASE 3: DATABASE ARCHITECTURE & EXPORT
# ==========================================
print("[3/4] Architecting SQL Database and Exporting Master Portfolio...")
conn = sqlite3.connect('nbfc_enterprise_system.db')
df_master.to_sql('Portfolio_Master', conn, if_exists='replace', index=False)
df_master.to_csv('nbfc_master_portfolio.csv', index=False)

# ==========================================
# PHASE 4: OPERATIONAL HITLIST EXTRACTION
# ==========================================
print("[4/4] Filtering Target Accounts for Field Operations...")
# Isolate High Risk, sort by danger level, and keep only actionable columns
df_hitlist = df_master[df_master['Risk_Category'] == 'High Risk (Intervene)'].sort_values(by='NPA_Probability', ascending=False)
actionable_columns = ['CustomerID', 'Vehicle_Model', 'Loan_Amount_INR', 'NPA_Probability', 'FOIR_Pct', 'CIBIL_Score']
df_hitlist_clean = df_hitlist[actionable_columns]

df_hitlist_clean.to_sql('Intervention_Hitlist', conn, if_exists='replace', index=False)
df_hitlist_clean.to_csv('nbfc_operational_hitlist.csv', index=False)
conn.close()

print("\n--- PIPELINE COMPLETE ---")
print(f"Total Master Records: {len(df_master)}")
print(f"Total High-Risk Hitlist Records: {len(df_hitlist_clean)}")

# Trigger Downloads if running in Google Colab
if IN_COLAB:
    print("\nTriggering file downloads...")
    files.download('nbfc_enterprise_system.db')
    files.download('nbfc_master_portfolio.csv')
    files.download('nbfc_operational_hitlist.csv')

INITIATING END-TO-END NBFC DATA PIPELINE...

[1/4] Synthesizing Demographic and Asset Data...
[2/4] Executing NPA Probability Algorithm...
[3/4] Architecting SQL Database and Exporting Master Portfolio...
[4/4] Filtering Target Accounts for Field Operations...

--- PIPELINE COMPLETE ---
Total Master Records: 1500
Total High-Risk Hitlist Records: 3

Triggering file downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>